# Exercise 2 — Guided transfer learning with CIFAR-10 — Solution

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
repo_root = Path.cwd().parents[1] if Path.cwd().name == "solutions" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(repo_root / "src"))
from cvis_ml.config import DatasetConfig
from cvis_ml.data import CIFAR10DataModule
from cvis_ml.models import TransferModelFactory, count_parameters, describe_trainable_parameters
from cvis_ml.engine import Trainer
from cvis_ml.visualization import show_batch, plot_history, show_confusion_matrix
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
conv = torch.nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1)
x = torch.randn(4, 3, 64, 64)
z = conv(x)
print("input shape :", tuple(x.shape))
print("output shape:", tuple(z.shape))
print("weight shape:", tuple(conv.weight.shape))

## Dataset

In [ ]:
cifar_cfg = DatasetConfig(
    name="cifar10",
    data_root=str(repo_root / "data_cache"),
    image_size=96,
    batch_size=32,
    max_train_samples=1500,
    max_val_samples=400,
    num_workers=2,
    seed=42,
)
cifar_dm = CIFAR10DataModule(
    root=cifar_cfg.data_root,
    image_size=cifar_cfg.image_size,
    batch_size=cifar_cfg.batch_size,
    num_workers=cifar_cfg.num_workers,
    max_train_samples=cifar_cfg.max_train_samples,
    max_val_samples=cifar_cfg.max_val_samples,
    seed=cifar_cfg.seed,
    augment=True,
)
cifar_data = cifar_dm.setup()
print("Classes:", cifar_data.class_names)
print("Number of classes:", cifar_data.num_classes)
print("Train batches:", len(cifar_data.train_loader))
print("Validation batches:", len(cifar_data.val_loader))
show_batch(cifar_data.train_loader, cifar_data.class_names, n=8)

## One model

In [ ]:
model = TransferModelFactory.create(
    architecture="resnet18",
    num_classes=cifar_data.num_classes,
    strategy="frozen",
    pretrained=True,
)
total_params, trainable_params = count_parameters(model)
print("Total parameters    :", total_params)
print("Trainable parameters:", trainable_params)
for row in describe_trainable_parameters(model):
    print(row)

In [ ]:
trainer = Trainer(model=model, device="auto", learning_rate=1e-3, weight_decay=1e-4)
result_resnet_frozen = trainer.fit(
    train_loader=cifar_data.train_loader,
    val_loader=cifar_data.val_loader,
    epochs=1,
    max_batches_per_epoch=20,
    name="resnet18_frozen",
)
plot_history(result_resnet_frozen.history, title="ResNet18 frozen")
show_confusion_matrix(result_resnet_frozen.y_true, result_resnet_frozen.y_pred, cifar_data.class_names, title="ResNet18 frozen")

## Baseline ladder

In [ ]:
experiments = [
    {"name": "resnet18_frozen", "architecture": "resnet18", "strategy": "frozen"},
    {"name": "resnet18_partial", "architecture": "resnet18", "strategy": "partial"},
    {"name": "mobilenet_v3_small_frozen", "architecture": "mobilenet_v3_small", "strategy": "frozen"},
]
all_results = []
for exp in experiments:
    print("\nRunning:", exp)
    model_i = TransferModelFactory.create(exp["architecture"], cifar_data.num_classes, exp["strategy"], pretrained=True)
    total, trainable = count_parameters(model_i)
    print(f"Parameters: total={total:,}, trainable={trainable:,}")
    trainer_i = Trainer(model_i, device="auto", learning_rate=1e-3 if exp["strategy"] == "frozen" else 1e-4, weight_decay=1e-4)
    result_i = trainer_i.fit(cifar_data.train_loader, cifar_data.val_loader, epochs=1, max_batches_per_epoch=20, name=exp["name"])
    all_results.append({
        "experiment": exp["name"], "architecture": exp["architecture"], "strategy": exp["strategy"],
        "total_params": total, "trainable_params": trainable,
        "val_accuracy": result_i.metrics["accuracy"], "val_macro_f1": result_i.metrics["macro_f1"],
        "runtime_s": result_i.metrics["runtime_s"],
    })
pd.DataFrame(all_results)

In [ ]:
answer_2_4 = """
A good first interpretation compares validation accuracy, macro-F1, runtime, and trainable parameter count rather than accuracy alone.
The frozen feature extractor usually trains fastest and has the fewest trainable parameters.
Partial fine-tuning may improve performance, but with a small subset and only one epoch it may not be clearly better.
If validation is noisy or the learning rate is not appropriate, partial fine-tuning can even look worse.
For an industrial first baseline, I would trust the simplest baseline that gives stable validation behavior and interpretable errors.
"""
print(answer_2_4)